# ConvLSTM — Integrated Pipeline

This notebook walks through the full ConvLSTM pipeline on the `integrate` branch.
Run cells in order. Sections 6–9 assume you run them sequentially.

In [ ]:
!pip install numpy pandas scikit-learn pyyaml matplotlib torch


## 1. Environment Check

In [ ]:
import sys, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"PyTorch {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  Device: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
except ImportError:
    print("PyTorch not installed yet — run the install cell above first")

try:
    import numpy as np; print(f"NumPy {np.__version__}")
    import pandas as pd; print(f"Pandas {pd.__version__}")
except ImportError as e:
    print(f"Missing: {e}")


## 2. Repo Paths & Imports

In [ ]:
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ── repo root ──
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── ConvLSTM dir ──
CONVLSTM_DIR = ROOT / "training" / "ConvLSTM"
if str(CONVLSTM_DIR) not in sys.path:
    sys.path.insert(0, str(CONVLSTM_DIR))

print(f"Repo root:     {ROOT}")
print(f"ConvLSTM dir:  {CONVLSTM_DIR}")

# ── pipeline imports ──
from model import ConvLSTMPredictor
from training.common.config import load_config, resolve_path
from training.common.data import chunk_specs, load_chunk, model_matrix_to_convlstm_frames, ChunkSpec
from training.common.metrics import absolute_and_squared_errors_dbm, denormalize
from training.common.results import append_metric_rows, load_band_definitions, output_dir
from training.common.windowing import aligned_history_matrix, selected_horizon_index, target_rows_for

print("All pipeline imports OK")

## 3. Data Preparation

The pipeline expects per-site CSV files under `evaluation/aerpaw/`.
If missing, this cell splits the merged 750-column CSV into per-site files.

In [ ]:
AERPAW_DIR = ROOT / "evaluation" / "aerpaw"
MERGED_CSV = ROOT / "training" / "data" / "merged_power_data_sub6GHz_avg_per_minute.csv"

AERPAW_DIR.mkdir(parents=True, exist_ok=True)

SITE_FILES = {
    "CC1": AERPAW_DIR / "ResultsCC1Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
    "CC2": AERPAW_DIR / "ResultsCC2Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
    "LW1": AERPAW_DIR / "ResultsLW1Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
}

missing = [n for n, p in SITE_FILES.items() if not p.exists()]
if missing:
    print(f"Generating missing site CSVs: {missing}")
    merged = np.loadtxt(MERGED_CSV, delimiter=",")
    print(f"Merged shape: {merged.shape}")
    for i, name in enumerate(["CC1", "CC2", "LW1"]):
        np.savetxt(SITE_FILES[name], merged[:, i*250:(i+1)*250],
                   delimiter=",", fmt="%.6f")
        print(f"  {name}: {SITE_FILES[name]} ({merged[:, i*250:(i+1)*250].shape})")
else:
    print("All site CSVs present:")
    for n, p in SITE_FILES.items():
        print(f"  {n}: {p.stem}.csv {np.loadtxt(p, delimiter=',').shape}")

## 4. Config Inspection

In [ ]:
config = load_config()

print("=== Data ===")
print(f"  data_dir:          {config['data']['data_dir']}")
print(f"  reference_site:    {config['data']['reference_site']}")
print(f"  test_splits:       {config['data']['test_splits']}")
print(f"  chunks:")
for c in config['data']['chunks']:
    print(f"    {c['id']}: {c['start_mhz']}-{c['end_mhz']} MHz")

print(f"\n=== Windowing ===")
print(f"  lookback:    {config['windowing']['lookback']}")
print(f"  horizons:    {config['windowing']['horizons']}")
print(f"  min_history: {config['windowing']['min_history']}")

print(f"\n=== ConvLSTM ===")
c = config['convlstm']
print(f"  epochs:              {c['epochs']}")
print(f"  batch_size:          {c['batch_size']}")
print(f"  learning_rate:       {c['learning_rate']}")
print(f"  input_seq_len:       {c['input_sequence_length']}")
print(f"  prediction_horizon:  {c['prediction_horizon']}")
print(f"  model:")
for k, v in c['model'].items():
    print(f"    {k}: {v}")

## 5. Single-Chunk Exploration & Plots

Load the first 200 MHz chunk and inspect its structure.

In [ ]:
chunks = chunk_specs(config)
chunk = chunks[0]
print(f"Chunk: {chunk.chunk_id} ({chunk.start_mhz}-{chunk.end_mhz} MHz)")

data = load_chunk(config, chunk)

print(f"Frequency bins: {len(data.shared_frequencies)}")
print(f"  range: {data.shared_frequencies[0]:.1f} – {data.shared_frequencies[-1]:.1f} MHz")
print(f"  step:  ~{data.shared_frequencies[1] - data.shared_frequencies[0]:.2f} MHz")
print(f"Normalization: {data.normalization}")
print(f"\nSplits:")
for name, sp in data.splits.items():
    print(f"  {name}: model_input {sp.model_input.shape}, raw_dbm {sp.raw_dbm.shape}")

In [ ]:
import matplotlib.pyplot as plt

train_raw = data.splits["CC2_train"].raw_dbm
n_bins = train_raw.shape[1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

# Time series for 3 bins
for idx in [0, n_bins//2, n_bins-1]:
    ax1.plot(train_raw[:, idx],
             label=f"{data.shared_frequencies[idx]:.1f} MHz", lw=0.8)
ax1.set_title(f"{chunk.chunk_id} — CC2 Train (raw dBm)")
ax1.set_ylabel("Power (dBm)")
ax1.legend(); ax1.grid(True, alpha=0.3)

# Spectrogram
im = ax2.imshow(train_raw.T, aspect="auto", cmap="viridis")
ax2.set_title(f"{chunk.chunk_id} — Train Spectrogram")
ax2.set_xlabel("Time (minutes)")
ax2.set_ylabel("Frequency bin")
plt.colorbar(im, ax=ax2, label="Power (dBm)")
plt.tight_layout(); plt.show()

## 6. Smoke Test — 1 Chunk, 5 Epochs

Train ConvLSTM on the first chunk inside the notebook for a quick sanity check.

In [ ]:
lookback = int(config["convlstm"].get("input_sequence_length", config["windowing"]["lookback"]))
prediction_horizon = int(config["convlstm"].get("prediction_horizon", max(config["windowing"]["horizons"])))
train = data.splits["CC2_train"].model_input
frames = model_matrix_to_convlstm_frames(train)
origins = np.arange(lookback - 1, len(frames) - prediction_horizon, dtype=np.int64)

print(f"Train shape: {train.shape}  →  {len(origins)} windows")

In [ ]:
class ConvLSTMWindowDataset(Dataset):
    def __init__(self, frames, lookback, prediction_horizon, origins):
        self.frames = torch.from_numpy(frames).float()
        self.lookback = lookback
        self.prediction_horizon = prediction_horizon
        self.origins = origins.astype(np.int64)
    def __len__(self): return len(self.origins)
    def __getitem__(self, idx):
        o = int(self.origins[idx])
        x = self.frames[o - self.lookback + 1 : o + 1].unsqueeze(1)
        y = self.frames[o + 1 : o + self.prediction_horizon + 1].unsqueeze(1)
        return x, y

val_frac, bs = 0.1, 32
val_cnt = max(1, int(len(origins) * val_frac))
train_o, val_o = origins[:-val_cnt], origins[-val_cnt:]

train_loader = DataLoader(ConvLSTMWindowDataset(frames, lookback, prediction_horizon, train_o),
                          batch_size=bs, shuffle=True, drop_last=True)
val_loader   = DataLoader(ConvLSTMWindowDataset(frames, lookback, prediction_horizon, val_o),
                          batch_size=bs, shuffle=False)

def model_config_fn(config, n_bins):
    ccfg = config["convlstm"]
    return dict(data=dict(n_nodes=1, n_bins_per_node=n_bins, node_names=["CC2"]),
                windowing=dict(input_sequence_length=lookback,
                               prediction_horizon=prediction_horizon),
                model=ccfg["model"])

model_config = model_config_fn(config, train.shape[1])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
model = ConvLSTMPredictor(model_config).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002, weight_decay=0.004)

EPOCHS = 5
clip_norm = 5.0
train_losses, val_losses = [], []

for ep in range(1, EPOCHS + 1):
    model.train(); tl = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, y_teacher=y, teacher_forcing_ratio=1.0), y)
        loss.backward()
        if clip_norm > 0: nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()
        tl += loss.item() * x.size(0)
    tl /= max(len(train_loader.dataset), 1)
    train_losses.append(tl)

    model.eval(); vl = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            vl += criterion(model(x), y).item() * x.size(0)
    vl /= max(len(val_loader.dataset), 1)
    val_losses.append(vl)
    print(f"Epoch {ep:02d}/{EPOCHS}  train={tl:.6f}  val={vl:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(train_losses, label="Train", marker=".")
ax.plot(val_losses,   label="Val",   marker=".")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
ax.set_title(f"Smoke Test — {chunk.chunk_id} ({EPOCHS} epochs)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Full Training (screen Session)

For actual model training, run the full pipeline in a detached `screen` session.
This avoids interruptions from notebook disconnect.

In [ ]:
full_cmd = (
    f"cd {ROOT}"
    " && source .venv/bin/activate"
    " && python3 training/ConvLSTM/train_integrated.py"
    " && deactivate"
)
screen_cmd = f"screen -dmS convlstm bash -c '{full_cmd}'"
print("Launch training in background:")
print(screen_cmd)
print()
print("Attach:  screen -r convlstm")
print("Detach:  Ctrl-A D")

## 8. Log Monitoring

After launching the screen session, monitor training logs here.

In [ ]:
import time
OUT_DIR = output_dir(config, "ConvLSTM")
log_path = OUT_DIR / "chunk_600_800_training_log.csv"

print(f"Waiting for {log_path} ...")
for _ in range(60):
    if log_path.exists():
        break
    time.sleep(2)
else:
    print("Log not found yet — training may still be starting.")

if log_path.exists():
    logs = pd.read_csv(log_path)
    display(logs.tail())

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(logs["train_loss"], label="Train", marker=".")
    ax.plot(logs["val_loss"],   label="Val",   marker=".")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 9. Post-Training Evaluation & Plots

Load the best checkpoint and evaluate on test splits. This uses the pipeline's own `train_integrated.py` functions.

In [ ]:
# ── reimport so we can load the checkpoint ──
from training.ConvLSTM.train_integrated import evaluate_chunk as full_evaluate_chunk

out = output_dir(config, "ConvLSTM")
bands = load_band_definitions(config)
print(f"Output dir: {out}")

In [ ]:
agg_rows, freq_rows, band_rows = [], [], []
for ch in chunk_specs(config):
    print(f"Evaluating {ch.chunk_id} ...")
    a, f, b = full_evaluate_chunk(config, ch, bands, out)
    agg_rows.extend(a); freq_rows.extend(f); band_rows.extend(b)

results = pd.DataFrame(agg_rows)
results.to_csv(out / "aggregate_metrics.csv", index=False)
results

In [ ]:
# Per-horizon bar chart
fig, ax = plt.subplots(figsize=(10, 4))
pivot = results.pivot_table(index="horizon", columns="chunk_id",
                            values="rmse_db", aggfunc="mean")
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("RMSE (dB)")
ax.set_title("ConvLSTM — RMSE by Horizon and Chunk")
ax.legend(title="Chunk")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# Prediction vs ground-truth sample plots
ch = chunks[0]
data = load_chunk(config, ch)
train = data.splits["CC2_train"].model_input
split = data.splits["CC2_test"]
full_x = np.vstack([train, split.model_input]).astype(np.float32)
full_raw = np.vstack([data.splits["CC2_train"].raw_dbm, split.raw_dbm]).astype(np.float32)
history_offset = len(train)
min_history = int(config["windowing"].get("min_history", 4320))
horizons = [int(h) for h in config["windowing"]["horizons"]]

bin_idx = n_bins // 2
freq = data.shared_frequencies[bin_idx]

fig, axes = plt.subplots(len(horizons), 1, figsize=(12, 3*len(horizons)), sharex=True)
for ax, horizon in zip(axes, horizons):
    target_rows = target_rows_for(len(split.raw_dbm), history_offset, horizon,
                                  lookback, min_history)
    origins = target_rows - horizon
    histories = aligned_history_matrix(full_x, origins, 0, lookback)
    histories = histories[:, :, None, None, :].astype(np.float32)
    loader = DataLoader(torch.from_numpy(histories).float(), batch_size=32, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for xb in loader:
            preds.append(model(xb.to(device))[:, selected_horizon_index(horizon), 0, 0, :].cpu().numpy())
    pred = np.concatenate(preds)
    t = target_rows - history_offset
    pred_dbm, _, _ = absolute_and_squared_errors_dbm(
        pred[:, bin_idx:bin_idx+1], full_raw[target_rows, bin_idx:bin_idx+1], data.normalization)
    ax.plot(t, split.raw_dbm[t, bin_idx], label="GT", lw=0.8)
    ax.plot(t, pred_dbm[:, 0], label="Pred", lw=0.8)
    ax.set_ylabel("dBm")
    ax.set_title(f"H={horizon} min — Bin {freq:.1f} MHz")
    ax.legend(); ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Test time (minutes)")
plt.tight_layout(); plt.show()